# Proyecto: Mercado de Limones Argentinos
## Notebook 02 — Pipeline Temporal y Análisis Comparativo
### Período: Enero 2023 — Marzo 2026
### Fecha de análisis: 13/06/2026
### Autor: Rodolfo Gabriel Riveros Lobos

---

## Objetivo del notebook

Consolidar los 7 archivos mensuales seleccionados en un único 
DataFrame y analizar la evolución temporal de los indicadores 
clave del mercado de transferencias automotrices.

## Diseño de la muestra

| Archivo | Período | Justificación |
|---|---|---|
| 2023_01 | Enero 2023 | Línea de base |
| 2023_10 | Octubre 2023 | Post-PASO, máxima incertidumbre electoral |
| 2024_02 | Febrero 2024 | Primer impacto desregulación Milei |
| 2024_12 | Diciembre 2024 | Cierre año, efecto principal-agente |
| 2025_01 | Enero 2025 | Record ventas 0km — mecanismo de transmisión |
| 2025_10 | Octubre 2025 | Mercado con nuevas marcas consolidadas |
| 2026_03 | Marzo 2026 | Crisis actual de sobrestock |

## Nota metodológica

Muestra de 7 puntos seleccionados por criterio teórico.
No es una serie continua. Las tendencias entre puntos 
son inferidas, no medidas directamente.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Configuración
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

# Ruta base de datos crudos
RAW_PATH = Path('../data/raw')
PROCESSED_PATH = Path('../data/processed')

print("Librerías cargadas correctamente.")

# Verificar que los 7 archivos están disponibles
archivos = sorted(RAW_PATH.glob('transferencias_*.csv'))
print(f"\nArchivos encontrados: {len(archivos)}")
for f in archivos:
    print(f"  {f.name} — {f.stat().st_size / 1024 / 1024:.1f} MB")

Librerías cargadas correctamente.

Archivos encontrados: 7
  transferencias_2023_01.csv — 34.9 MB
  transferencias_2023_10.csv — 37.0 MB
  transferencias_2024_02.csv — 23.3 MB
  transferencias_2024_12.csv — 35.3 MB
  transferencias_2025_01.csv — 37.4 MB
  transferencias_2025_10.csv — 36.7 MB
  transferencias_2026_03.csv — 33.9 MB


---
## Pipeline de carga y limpieza

La función `cargar_y_limpiar()` encapsula todo el proceso 
del notebook 01_exprolarcion.ipynb en un bloque reutilizable.

Decisiones de limpieza aplicadas a cada archivo:
1. Encoding utf-8-sig — resuelve BOM
2. Exclusión SUBASTADOS/CLÁSICOS
3. Exclusión nulos en automotor_anio_modelo
4. Exclusión años fuera de rango 1950 — año del archivo
5. Construcción de variable antigüedad
6. Construcción de variable rango_antiguedad
7. Agregado de columna periodo para identificar el archivo

In [2]:
def cargar_y_limpiar(filepath):
    """
    Carga y limpia un archivo mensual de transferencias DNRPA.
    Retorna un DataFrame limpio con variables construidas.
    """
    
    # Extraer año y mes del nombre del archivo
    nombre = Path(filepath).stem  # transferencias_2023_01
    partes = nombre.split('_')
    anio = int(partes[1])
    mes = int(partes[2])
    periodo = f"{anio}-{mes:02d}"
    
    # Carga
    df = pd.read_csv(
        filepath,
        encoding='utf-8-sig',
        sep=',',
        low_memory=False,
        dtype={'automotor_tipo_codigo': str}
    )
    
    registros_originales = len(df)
    
    # Exclusión subastados y clásicos
    df = df[df['tramite_tipo'] != 'TRANSFERENCIA SUBASTADOS / AFF / CLASICOS'].copy()
    
    # Exclusión nulos en anio_modelo
    df = df.dropna(subset=['automotor_anio_modelo'])
    
    # Conversión y filtro de rango válido
    df['automotor_anio_modelo'] = pd.to_numeric(
        df['automotor_anio_modelo'], errors='coerce'
    ).astype('Int64')
    
    df = df[
        (df['automotor_anio_modelo'] >= 1950) &
        (df['automotor_anio_modelo'] <= anio)
    ].copy()
    
    # Variable antigüedad
    df['antiguedad'] = anio - df['automotor_anio_modelo']
    
    # Variable rango_antiguedad
    bins = [0, 3, 7, 15, 25, 40, 75]
    labels = ['0-3 años', '4-7 años', '8-15 años',
              '16-25 años', '26-40 años', '41+ años']
    df['rango_antiguedad'] = pd.cut(
        df['antiguedad'],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    
    # Columnas de identificación temporal
    df['periodo'] = periodo
    df['anio_archivo'] = anio
    df['mes_archivo'] = mes
    
    registros_finales = len(df)
    
    print(f"{periodo} → {registros_originales:>7,} originales "
          f"→ {registros_finales:>7,} limpios "
          f"({registros_finales/registros_originales*100:.1f}%)")
    
    return df

In [3]:
# Ejecutar pipeline sobre los 7 archivos
print("CARGANDO Y LIMPIANDO ARCHIVOS")
print("=" * 60)

dataframes = []

for filepath in sorted(RAW_PATH.glob('transferencias_*.csv')):
    df_mes = cargar_y_limpiar(filepath)
    dataframes.append(df_mes)

# Consolidar en único DataFrame
df_total = pd.concat(dataframes, ignore_index=True)

print("=" * 60)
print(f"\nDataFrame consolidado:")
print(f"  Shape total:  {df_total.shape}")
print(f"  Períodos:     {sorted(df_total['periodo'].unique())}")
print(f"  Memoria:      {df_total.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

CARGANDO Y LIMPIANDO ARCHIVOS
2023-01 → 133,102 originales → 132,129 limpios (99.3%)
2023-10 → 141,592 originales → 140,402 limpios (99.2%)
2024-02 → 106,239 originales → 105,408 limpios (99.2%)
2024-12 → 160,670 originales → 159,667 limpios (99.4%)
2025-01 → 170,105 originales → 169,213 limpios (99.5%)
2025-10 → 166,670 originales → 165,683 limpios (99.4%)
2026-03 → 154,230 originales → 153,351 limpios (99.4%)

DataFrame consolidado:
  Shape total:  (1025853, 30)
  Períodos:     ['2023-01', '2023-10', '2024-02', '2024-12', '2025-01', '2025-10', '2026-03']
  Memoria:      1449.1 MB


---
## Optimización de memoria

El DataFrame consolidado ocupa 1.449 MB en memoria.
Antes de proceder con el análisis reducimos el footprint
convirtiendo columnas categóricas a tipo `category` 
y eliminando columnas no utilizadas en este análisis.

In [4]:
# Columnas que usamos en el análisis temporal
COLUMNAS_UTILES = [
    'periodo',
    'anio_archivo',
    'mes_archivo',
    'tramite_tipo',
    'automotor_origen',
    'automotor_anio_modelo',
    'automotor_marca_descripcion',
    'automotor_tipo_descripcion',
    'registro_seccional_provincia',
    'titular_tipo_persona',
    'titular_genero',
    'titular_anio_nacimiento',
    'antiguedad',
    'rango_antiguedad'
]

# Reducir a columnas útiles
df_total = df_total[COLUMNAS_UTILES].copy()

# Convertir columnas categóricas
categoricas = [
    'periodo',
    'tramite_tipo',
    'automotor_origen',
    'automotor_marca_descripcion',
    'automotor_tipo_descripcion',
    'registro_seccional_provincia',
    'titular_tipo_persona',
    'titular_genero',
    'rango_antiguedad'
]

for col in categoricas:
    df_total[col] = df_total[col].astype('category')

# Reporte de optimización
memoria_final = df_total.memory_usage(deep=True).sum() / 1024 / 1024

print(f"Memoria antes: 1,449.1 MB")
print(f"Memoria después: {memoria_final:.1f} MB")
print(f"Reducción: {(1 - memoria_final/1449.1)*100:.1f}%")
print(f"\nShape: {df_total.shape}")
print(f"\nDtypes optimizados:")
print(df_total.dtypes)

Memoria antes: 1,449.1 MB
Memoria después: 96.5 MB
Reducción: 93.3%

Shape: (1025853, 14)

Dtypes optimizados:
periodo                         category
anio_archivo                       int64
mes_archivo                        int64
tramite_tipo                    category
automotor_origen                category
automotor_anio_modelo              Int64
automotor_marca_descripcion     category
automotor_tipo_descripcion      category
registro_seccional_provincia    category
titular_tipo_persona            category
titular_genero                  category
titular_anio_nacimiento           object
antiguedad                         Int64
rango_antiguedad                category
dtype: object


In [5]:
# Corregir titular_anio_nacimiento
df_total['titular_anio_nacimiento'] = pd.to_numeric(
    df_total['titular_anio_nacimiento'], 
    errors='coerce'
).astype('Int64')

print(f"titular_anio_nacimiento dtype: {df_total['titular_anio_nacimiento'].dtype}")
print(f"Nulos: {df_total['titular_anio_nacimiento'].isnull().sum():,}")
print(f"Rango: {df_total['titular_anio_nacimiento'].min()} "
      f"— {df_total['titular_anio_nacimiento'].max()}")

titular_anio_nacimiento dtype: Int64
Nulos: 56
Rango: 1900 — 9999


In [6]:
# Guardar antes de las visualizaciones
# Si el notebook crashea, no reprocesamos todo
output_path = PROCESSED_PATH / 'transferencias_consolidado.csv'

df_total.to_csv(output_path, index=False, encoding='utf-8')

print(f"Dataset consolidado guardado.")
print(f"Ruta: {output_path}")
print(f"Shape: {df_total.shape}")

Dataset consolidado guardado.
Ruta: ..\data\processed\transferencias_consolidado.csv
Shape: (1025853, 14)
